# Coders of Panama 🇵🇦
### A Panama Tech Academy Python Challenge

You walk into the glass tower office of **Rodrigo Castillo**, the most famous (and most impatient) venture capitalist in Panama City, and ask him for **2.2 billion USD** to build "the Silicon Valley of Central America."

He laughs so loudly the whole floor turns around.

Then he asks where you live. *"Casco Viejo,"* you say. He leans back, thinks for a moment, and gives you a challenge instead:

> Collect the raw Instagram data of all **@panamatechacademy** followers and answer these questions:

- Who has the **maximum posts**?
- Who has the **maximum followers**?
- Who **follows the maximum** number of people?
- **How many categories** (Digital creator, Nonprofit organization, Community, etc.) are there, and **how many people** are in each?

You have **24 hours**.

You ask him for the data. He laughs again!

# Data Collection
You hire **Valentina Ríos** and **Samuel Chen**, who live in **Costa del Este** and **San Miguelito**.

Costa del Este and San Miguelito are far apart (and the Corredor at rush hour is no joke), so you set up a meeting at a little café in **Casco Viejo**.

Together they copy every follower profile into a plain text file, `finaldata.txt`. Each profile is a block separated by a blank line:

```
username
<n> posts
<n> followers        (can be 6,851 / 22.4K / 1.2M)
<n> following
display name
category             (optional)
bio lines...         (optional)
```

An early sample they collected is in `initialdata.txt`.

## Step 1 — Read the raw file

In [1]:
with open("finaldata.txt", encoding='utf-8') as f:
    data = f.read()

## Step 2 — Split it into one chunk per profile
Profiles are separated by blank lines. We drop empty / tiny leftovers.

In [2]:
chunks = data.split("\n\n")
chunks = [c for c in chunks if len(c) > 3]
print(f"Profiles found: {len(chunks)}")

Profiles found: 107


## Step 3 — Parse each chunk into a dictionary
Counts like `22.4K` or `1.2M` are converted to real integers. If a profile has no category line, it is labelled `Unknown`.

In [3]:
def to_number(text, word):
    value = text.split(word)[0].replace(",", "").strip()
    if "K" in value:
        return int(float(value.replace("K", "")) * 1000)
    if "M" in value:
        return int(float(value.replace("M", "")) * 1000000)
    return int(float(value))


def parse_chunk(chunk):
    sep_chunk = chunk.strip().split('\n')
    username = sep_chunk[0]
    no_of_posts = to_number(sep_chunk[1], " post")
    no_of_followers = to_number(sep_chunk[2], " follower")
    no_of_following = to_number(sep_chunk[3], " following")
    name = sep_chunk[4]
    if len(sep_chunk) > 5:
        type_of_page = sep_chunk[5]
        bio = "\n".join(sep_chunk[6:])
    else:
        type_of_page = "Unknown"
        bio = ""
    return {"username": username, "no_of_posts": no_of_posts, "no_of_followers": no_of_followers,
            "type_of_page": type_of_page, "no_of_following": no_of_following, "name": name, "bio": bio}

In [4]:
all_chunks = []
for chunk in chunks:
    parsed_chunk = parse_chunk(chunk)
    all_chunks.append(parsed_chunk)

all_chunks[0]

{'username': 'techie_casco',
 'no_of_posts': 420,
 'no_of_followers': 9800,
 'type_of_page': 'Software Engineer',
 'no_of_following': 310,
 'name': 'Andrés M.',
 'bio': '💻 Full-stack @ SaaS startup\n☕ Working from cafes in Casco Viejo\ngithub.com/andresmakes'}

## Step 4 — Save the clean data as JSON
Now anyone at Panama Tech Academy can reuse the data without parsing it again.

In [5]:
import json
s = json.dumps(all_chunks, indent=4)
with open("data.json", "w", encoding="utf-8") as f:
    f.write(s)
print(f"Saved {len(all_chunks)} profiles to data.json")

Saved 107 profiles to data.json


# Who has the maximum posts?
Let's write code to find max posts.

In [6]:
max_posts = 0
for chunk in all_chunks:
    if max_posts < chunk['no_of_posts']:
        max_posts = chunk['no_of_posts']
        chunk_with_max_post = chunk
print(chunk_with_max_post['username'], "-", chunk_with_max_post['no_of_posts'], "posts")

startuphub_pty - 2300 posts


# Who has the maximum followers?
Let's write code to find who has max followers.

In [7]:
max_followers = 0
for chunk in all_chunks:
    if max_followers < chunk['no_of_followers']:
        max_followers = chunk['no_of_followers']
        chunk_with_max_followers = chunk
print(chunk_with_max_followers['username'], "-", chunk_with_max_followers['no_of_followers'], "followers")

_lucianavega - 681000 followers


# Who follows maximum people?
Let's write code to find who follows maximum people.

In [8]:
max_following = 0
for chunk in all_chunks:
    if max_following < chunk['no_of_following']:
        max_following = chunk['no_of_following']
        chunk_with_max_following = chunk
print(chunk_with_max_following['username'], "-", chunk_with_max_following['no_of_following'], "following")

panama_tech_bro - 890 following


# How many categories?
Let's write code to find out the number of categories.

In [9]:
categories = set()
for chunk in all_chunks:
    categories.add(chunk['type_of_page'])
print(f"There are {len(categories)} categories")

There are 34 categories


# How many people are in each category?
The second half of Rodrigo's question — count the profiles per category, biggest first.

In [10]:
category_count = {}
for chunk in all_chunks:
    category = chunk['type_of_page']
    category_count[category] = category_count.get(category, 0) + 1

for category, count in sorted(category_count.items(), key=lambda item: item[1], reverse=True):
    print(f"{category:<25} {count}")

Community                 33
Media                     12
Blog                      8
Tech                      7
Personal Blog             5
Education                 4
Lifestyle                 3
Tech Creator              2
Engineer                  2
Food & Drink              2
Developer                 2
Design                    2
Entrepreneur              2
Public Figure             2
Educator                  2
Software Engineer         1
Cafe Explorer             1
Founder                   1
Backend Developer         1
Frontend Dev              1
Full-stack Dev            1
Artist + Coder            1
Product Manager           1
Recruiter                 1
Cafe Nerd                 1
Coder                     1
Comedy                    1
Data Science              1
Data Scientist            1
Nonprofit organization    1
Digital creator           1
Blogger                   1
Investor                  1
Writer                    1


# Bonus — Data quality check
Two people collected the data, so some profiles may have been copied twice. Let's look for repeated usernames.

In [11]:
seen = {}
for chunk in all_chunks:
    seen[chunk['username']] = seen.get(chunk['username'], 0) + 1

duplicates = [u for u, c in seen.items() if c > 1]
print("Total profiles:", len(all_chunks))
print("Unique usernames:", len(seen))
print("Duplicated usernames:", duplicates)

Total profiles: 107
Unique usernames: 105
Duplicated usernames: ['sanfran_startup_stories', 'machinelearning_pty']


# Summary for Rodrigo

| Question | Answer |
|---|---|
| Maximum posts | **startuphub_pty** (2,300 posts) |
| Maximum followers | **_lucianavega** (681,000 followers) |
| Follows the most people | **panama_tech_bro** (890 following) |
| Number of categories | **34** (see counts above — *Community* is the largest) |

Challenge completed in well under 24 hours. Now about that 2.2 billion... 😄

*— Panama Tech Academy*